In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [1]:
!pip install sentence-transformers -q
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


In [2]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [3]:
# Load pretrained model - good balance of speed and quality
model = SentenceTransformer('all-MiniLM-L6-v2')

def predict_top3(row, model):
    prompt = str(row['prompt'])
    options = {opt: str(row[opt]) for opt in ['A', 'B', 'C', 'D', 'E']}
    
    # Encode prompt and all options
    prompt_emb = model.encode([prompt])
    option_embs = model.encode(list(options.values()))
    
    # Cosine similarity between prompt and each option
    sims = cosine_similarity(prompt_emb, option_embs)[0]
    
    # Rank options by similarity
    ranked = sorted(zip(options.keys(), sims), key=lambda x: x[1], reverse=True)
    top3 = [opt for opt, _ in ranked[:3]]
    return ' '.join(top3)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# checking MAP@3 on training data before submitting!
def map_at_3(correct, predictions):
    for k, pred in enumerate(predictions[:3]):
        if pred == correct:
            return 1.0 / (k + 1)
    return 0.0

scores = []
for _, row in train.iterrows():
    pred_str = predict_top3(row, model)
    top3 = pred_str.split()
    scores.append(map_at_3(row['answer'], top3))

print(f"Sentence Transformer MAP@3 on train: {round(sum(scores)/len(scores), 4)}")

Sentence Transformer MAP@3 on train: 0.4231


In [4]:
# Run predictions on test set
print("Generating predictions...")
predictions = []
for i, (_, row) in enumerate(test.iterrows()):
    pred = predict_top3(row, model)
    predictions.append({'id': row['id'], 'prediction': pred})
    if i % 100 == 0:
        print(f"  {i}/{len(test)} done...")

submission = pd.DataFrame(predictions)
submission.to_csv('submission.csv', index=False)
print("Done!", submission.head())

Generating predictions...
  0/500 done...
  100/500 done...
  200/500 done...
  300/500 done...
  400/500 done...
Done!    id prediction
0   1      B E A
1   2      B D C
2   3      A C D
3   4      A E C
4   5      B C D


In [ ]:
'''
This submission uses a similarity-based approach to predict answers.

In this method, the question and all five answer options are converted into numerical vectors using a pre-trained language model. 
After that, cosine similarity is calculated between the question and each answer option.

The options with the highest similarity scores are considered more relevant to the question and are ranked accordingly. 
The top three ranked options are then used as the final prediction.

This approach is different from the TF-IDF method used earlier as TF-IDF mainly checks common words between the question and answer, 
while this method tries to compare the overall meaning of the texts.

By using this method, the leaderboard score obtained was : 0.38653
This score is higher than the TF-IDF baseline, which shows that comparing the meaning of the question and answers works better than simply matching words.

The Conclusion: This approach gives better results than the previous baseline methods. However, the score shows that similarity alone is not enough to solve all questions correctly, 
especially those that require deeper reasoning and knowledge.

'''